# Load a diffusion-generated MNIST classifier

Run the locked evaluation first, then select the project `.venv` as the notebook kernel. This example loads the candidate selected by **validation** accuracy and predicts on twelve validation images. It performs no optimization. The final report compares every generated candidate, including poor samples.


In [ ]:
from pathlib import Path
import json
import torch
import matplotlib.pyplot as plt
from torchvision.datasets import MNIST
from torchvision.transforms import ToTensor
from p_diff.classifier import Classifier

root = Path.cwd()
if not (root / "src" / "p_diff").exists():
    root = root.parent
assert (root / "src" / "p_diff").exists(), "Run from the project root or notebooks folder"
torch.set_num_threads(1)


## Load the validation-selected candidate

The selection file was sealed before official-test evaluation. This example follows that saved choice; it never chooses a model from test accuracy.


In [ ]:
evaluation = root / "artifacts" / "locked-evaluation"
selection = json.loads((evaluation / "selections.json").read_text())["selected"]["diffusion"]
checkpoint = torch.load(evaluation / selection["file"], weights_only=True)
model = Classifier().eval()
model.load_state_dict(checkpoint["state_dict"], strict=True)
print(f"Diffusion candidate seed: {selection["seed"]}")
print(f"Selection validation accuracy: {selection["validation_accuracy"]:.2%}")


## Predict immediately

Images use the experiment preprocessing: float32 values in [0, 1], shape `[batch, 1, 28, 28]`. Prediction needs only the saved classifier; MNIST is loaded here to supply example images.


In [ ]:
splits = torch.load(root / "artifacts" / "source-full" / "splits.pt", weights_only=True)
dataset = MNIST(root / "data", train=True, download=False, transform=ToTensor())
examples = [dataset[int(index)] for index in splits["validation"][:12]]
images = torch.stack([image for image, label in examples])
labels = torch.tensor([label for image, label in examples])
with torch.no_grad():
    predictions = model(images).argmax(dim=1)
print("Predictions:", predictions.tolist())
print("Labels:", labels.tolist())


In [ ]:
fig, axes = plt.subplots(2, 6, figsize=(10, 4), layout="constrained")
for i, axis in enumerate(axes.flat):
    axis.imshow(images[i, 0], cmap="gray", vmin=0, vmax=1)
    correct = predictions[i] == labels[i]
    axis.set_title(f"Pred {int(predictions[i])} / true {int(labels[i])}", color="green" if correct else "red", fontsize=10)
    axis.axis("off")
plt.show()


## Interpretation

These twelve examples are a usage demonstration, not an accuracy estimate. Read `plans/06_Locked_Evaluation_Results.md` for the complete locked evaluation, cost limits, and baseline comparisons. No generated classifier is fine-tuned after loading.
